| Trial | Factual Test | Conceptual Test | Technical Test |
| -------- | ------- | --- | --- |
| A (Naive) | 3 - Answered the question, but little to no diversity. | 2 - Did not answer the question from a larger perspective, only pulled from personal accounts of the event. | 2 - Barely answered the question, draws from mostly the same sources again	|
| B (MMR )| 3 - answers question somewhat well, much diversity | 4 - great diversity, answered question | 2 - Poorly answered question, but diverse sources although wrong |
| C (Hybrid)| 5 - great diversity and answered question | 4 - Good divesity, mostly answered question | 5 - Great Diversity, perfectly answered question with great detail
 |

 ANALYSIS:


1.   Trial A returned multiple repeated chunks, whereas Trail B did not, and Trial B also returned a larger picture for the conceptual question as it was able to pull more diverse chuncks that could answer the question.
2.   Hybrid did find things that the vector-only searches missed, such as the use of "glider troops" and the english channel.
3. The LLM did admit to not having some information, such as my name, and did not hallucinate. Retrieval quality impacts groundedness because a good reteriver will be less likely to hallucinate, and as the LLM result will output the best answer available to it rather than something that is outright false or redundant, making it grounded.
4. I would recommend the hybrid approach for discussions about the Normandy Invasion as there are many specific details that need to be picked up on while also being able to grasp larger concepts, which the naive approach can not do either, and MMR being only good for diversity in concepts and choices. That is why I believe hybrid outperformed the other apporaches.




In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [2]:
# Updated for February 2026 stability
!pip install -qU \
    langchain \
    langchain-core \
    langchain-community \
    langchain-google-genai \
    langchain-chroma \
    langchain-classic \
    langchain-text-splitters \
    pypdf


import os
from google.colab import userdata

# API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.2/504.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2

In [17]:
!pip install -qU langchain-core langchain-community langchain-google-genai langchain-classic
!pip install -qU faiss-cpu pypdf rank_bm25 flashrank

# Standard imports for environment management
import os
from google.colab import userdata

# API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.5 MB/s eta 0:00:00


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Updated list to match your specific filenames
file_names = [
    "WWII-Invasion-of-Normandy-Resource-Packet.pdf",
    "wie175.pdf",
    "WM_-_Dick_Robert.pdf",
    "Dday_and_the_RAF.pdf",
    "d-day-brief-history.pdf"
]

all_docs = []

# Loop through each file, load its content, and split it into chunks
for file in file_names:
    if os.path.exists(file):
        print(f"Processing {file}...")
        loader = PyPDFLoader(file)

        # Recursive splitter attempts to keep paragraphs and sentences together
        # chunk_size is total characters; overlap keeps context across boundaries
        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=15)
        chunks = loader.load_and_split(splitter)

        all_docs.extend(chunks)
        splits = splitter.split_documents(all_docs)
    else:
        print(f"Warning: '{file}' not found in the sidebar. Please upload it.")

if all_docs:
    print(f"\nSuccess! Total text chunks created: {len(all_docs)}")

Processing WWII-Invasion-of-Normandy-Resource-Packet.pdf...
Processing wie175.pdf...
Processing WM_-_Dick_Robert.pdf...
Processing Dday_and_the_RAF.pdf...
Processing d-day-brief-history.pdf...

Success! Total text chunks created: 54


In [30]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Initialize Gemini Embeddings
# The model "models/gemini-embedding-001" is the standard for Gemini vectors
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")






# Create the vector store
# Using the newer 'langchain_chroma' package
vector_store = Chroma(embedding_function=embeddings)



# Add documents with explicit IDs
vector_store.add_documents(all_docs)




# Similarity Search
results = vector_store.similarity_search("What day did D-Day occur?", k=5)




# MMR (Max Marginal Relevance) logic
found_docs = vector_store.similarity_search("retrieval", k=5)
print(f"Found documents: {len(found_docs)}")

used_doc_ids = {doc.metadata["id"] for doc in found_docs}
remaining_docs = [doc for doc in all_docs if doc.metadata["id"] not in used_doc_ids]
print(f"Remaining documents available for MMR: {len(remaining_docs)}")

if len(remaining_docs) > 0:
    # MMR focuses on finding documents that are relevant but also different
    # from what was already found, to avoid repetitive answers.
    mmr_results = vector_store.max_marginal_relevance_search(
        "retrieval systems",
        k=5,
        fetch_k=20, # fetch_k should generally be >= k
        lambda_mult=0.0
    )
    print(f"MMR Result: {mmr_results[0].page_content}")




GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 44.603942245s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDim

In [8]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma



# Initialize the 2026 flagship embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="RETRIEVAL_DOCUMENT",
    persist_directory="./chroma_db" # Added for consistency
)



# Initialize Chroma (using the modular 2026 syntax)
vectorstore = Chroma(
    collection_name="digitate_report_2026",
    embedding_function=embeddings
)


# Clear any old data
ids = vectorstore.get()['ids']
if ids:
    vectorstore.delete(ids=ids)
    print(f"Cleared {len(ids)} old entries from the vectorstore.")



# Batch upload with rate-limit protection (Mandatory for Free Tier)
batch_size = 50
print(f" Embedding {len(splits)} chunks...")

for i in range(0, len(splits), batch_size):
    batch = splits[i : i + batch_size]
    vectorstore.add_documents(batch)
    print(f" Indexed chunks {i} to {min(i + batch_size, len(splits))}")
    if i + batch_size < len(splits):
        time.sleep(15)



# --- RETRIEVER TEST SECTION ---

# Convert the vectorstore into a retriever
# k=4 remains the 2026 standard for high-density reports
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})



# Execute a test query
query = "How was the Whermacht defeated by the combination of allied  littoral, air, and infantry power?"
results = retriever.invoke(query)




# Print the results with enhanced Metadata (source tracking) formatting
print(f"\n Testing Retriever with query: '{query}'")

for i, doc in enumerate(results):
    # PyPDFLoader (pypdf) defaults to 0-indexed 'page'
    page_num = doc.metadata.get('page', 'Unknown')
    display_page = page_num + 1 if isinstance(page_num, int) else "Unknown"

    print(f"\n--- Result {i+1} | Source: Page {display_page} ---")

    # CLEANING: Replace multiple newlines/tabs with a single space for readability
    clean_content = " ".join(doc.page_content.split())

    # SNIPPET: Showing 350 chars (slightly more for 2026 technical analysis)
    print(f"Content: {clean_content[:350]}...")

    # OPTIONAL: Print the specific source file if metadata exists
    source_file = doc.metadata.get('source', 'Unknown File')
    if source_file != 'Unknown File':
        print(f"File: {source_file.split('/')[-1]}")

 Embedding 54 chunks...
 Indexed chunks 0 to 50
 Indexed chunks 50 to 54

 Testing Retriever with query: 'How was the Whermacht defeated by the combination of allied  littoral, air, and infantry power?'

--- Result 1 | Source: Page 1 ---
Content: the Americans suffered more than 2,000 casualties. By nightfall nearly all the Allied soldiers were ashore at a cost of 10,000 American, British, and Canadian casualties. Hitler’s vaunted Atlantic Wall had been breached in less than one da y. The beaches were secure, but it would take many weeks before the Allies could fight their way out of the he...
File: d-day-brief-history.pdf

--- Result 2 | Source: Page 2 ---
Content: “We just had to . . . try to get to the bottom of the cliffs on which the Germans had mounted their defenses.” By midday, the Americans had surmounted the cliffs and taken Omaha Beach at a heavy cost: over 4,700 killed, wounded, or missing out of the total of approximately 35,000 who came ashore that day, a loss rate of mor

In [19]:
import time
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Initialize Gemini Embeddings (2026 Stable Model)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)



# Define the Retry-Safe Embedding Wrapper
# This function will wait and retry automatically if we get a 429 error.
@retry(
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(Exception) # Catch the GoogleGenerativeAIError
)
def embed_with_retry(vector_store, batch):
    if vector_store is None:
        return FAISS.from_documents(batch, embeddings)
    else:
        vector_store.add_documents(batch)
        return vector_store



# Process with Batching and Retries
batch_size = 15 # Smaller batches help prevent hitting the limit too fast
vectorstore = None

print(f"Embedding {len(all_docs)} chunks with rate-limit protection...")
for i in range(0, len(all_docs), batch_size):
    batch = all_docs[i : i + batch_size]
    try:
        vectorstore = embed_with_retry(vectorstore, batch)
        print(f" Processed {i + len(batch)}/{len(all_docs)}...")
    except Exception as e:
        print(f" Failed after retries: {e}")
        break
    time.sleep(3) # A steady 3-second heartbeat to keep the API happy




# Finalize Retrievers
if vectorstore:
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    bm25_retriever = BM25Retriever.from_documents(all_docs)
    bm25_retriever.k = 5

    hybrid_retriever = EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[0.5, 0.5]
    )
    print("\n Hybrid Retriever is ready!")



Embedding 54 chunks with rate-limit protection...
 Processed 15/54...
 Processed 30/54...
 Processed 45/54...
 Processed 54/54...

 Hybrid Retriever is ready!


In [21]:
# UPDATED 2026 IMPORTS
try:
    # Most stable 2026 path for Contextual Compression
    from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
    from langchain.retrievers.document_compressors.flashrank_rerank import FlashrankRerank
except ImportError:
    # Alternative path used in some v1.x sub-versions
    from langchain_classic.retrievers import ContextualCompressionRetriever
    from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank



# Initialize FlashRank (The tiny but mighty re-ranker)
# We set top_n=3 to ensure the LLM only sees the most relevant facts.
compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2", top_n=3)



# Create the Compression Retriever
# This wraps the Hybrid Retriever we just successfully built.
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever
)



#  Final Test of the Pipeline
query = "Why did the allies invade normandy beach?"
final_docs = compression_retriever.invoke(query)

print(f" Re-ranker initialized and tested. Retrieved {len(final_docs)} chunks.")



# Display the high-quality results
for i, doc in enumerate(final_docs):
    print(f"\nRANKED RESULT {i+1} (Source: {doc.metadata.get('source')}):")
    print(f"{doc.page_content[:450]}...")



 Re-ranker initialized and tested. Retrieved 3 chunks.

RANKED RESULT 1 (Source: d-day-brief-history.pdf):
struggling for a common goal—the defeat of Nazi Ger many.  Because of highly intricate 
deception plans, Hitler and most of his staff belie ved that the Allies would be attacking at the 
Pas-de-Calais, the narrowest point between Great Britain and France.  
 
In the early morning darkness of June 6, thousands of 
Allied paratroopers and glider troops landed silent ly behind 
enemy lines, securing key roads and bridges on the flanks 
of the invasi...

RANKED RESULT 2 (Source: WWII-Invasion-of-Normandy-Resource-Packet.pdf):
1. Why did Germany have control of Normandy, France in June 1944? How long 
had they occupied the territory?
2. What defensive measures had the Germans utilized on the beaches to prevent 
an Allied invasion?
3. When did the troops learn of their planned invasion? When was the invasion 
originally planned for, and why did it not occur?
4. How did veterans of the g

In [24]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Hub import for modular LangChain
try:
    from langchain_classic import hub
except ImportError:
    import langchainhub as hub


# Gemini 2.5 Flash is the 'sweet spot' for speed and availability.
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1
)



# Attach Rate-Limit Protection
llm_with_retry = llm.with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True
)



# Pull the standard RAG prompt
prompt = hub.pull("rlm/rag-prompt")



# THE LCEL PIPELINE (The "Pipe" Syntax)
rag_chain = (
    RunnableParallel({
        "context": compression_retriever,
        "question": RunnablePassthrough()
    })
    | prompt
    | llm_with_retry
    | StrOutputParser()
)



# Final Execution
print("--- Final RAG Project Test (Gemini 2.5 Flash) ---\n")
query = "What is my name?"

try:
    answer = rag_chain.invoke(query)
    print(f"QUESTION: {query}")
    print(f"\nANSWER:\n{answer}")
except Exception as e:
    print(f" Chain failed: {e}")



--- Final RAG Project Test (Gemini 2.5 Flash) ---

QUESTION: What is my name?

ANSWER:
I don't know your name. The provided context discusses the D-Day invasion and mentions individuals like General Dwight D. Eisenhower and Dick Robert, but it does not contain any information about your identity.
